In [10]:
import torch
import pandas as pd
from transformers import MT5ForConditionalGeneration, MT5Tokenizer
import evaluate
from tqdm import tqdm

In [11]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [12]:
model = MT5ForConditionalGeneration.from_pretrained(
    "mt5-khmer-summary"
).to(DEVICE)

In [13]:
model

MT5ForConditionalGeneration(
  (shared): Embedding(250112, 512)
  (encoder): MT5Stack(
    (embed_tokens): Embedding(250112, 512)
    (block): ModuleList(
      (0): MT5Block(
        (layer): ModuleList(
          (0): MT5LayerSelfAttention(
            (SelfAttention): MT5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): MT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): MT5LayerFF(
            (DenseReluDense): MT5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
          

In [15]:
import pandas as pd
import csv

df = pd.read_csv(
    "data/test.csv",
    engine="python",
    encoding="utf-8",
    quoting=csv.QUOTE_NONE,
    on_bad_lines="skip"
)

print("Loaded rows:", len(df))
print(df.columns)

Loaded rows: 1055
Index(['title', 'author', 'date', 'content', 'summary', 'category', 'url',
       'source'],
      dtype='object')


In [16]:
tokenizer = MT5Tokenizer.from_pretrained("mt5-khmer-summary")

rouge = evaluate.load("rouge")

predictions = []
references = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    article = "summary: " + str(row["content"])
    reference = str(row["summary"])

    inputs = tokenizer(
        article,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=256,
            num_beams=8,
            early_stopping=True
        )

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)

    predictions.append(pred)
    references.append(reference)

results = rouge.compute(
    predictions=predictions,
    references=references
)

print("ROUGE RESULTS")
for k, v in results.items():
    print(f"{k}: {v:.4f}")

100%|██████████| 1055/1055 [06:28<00:00,  2.72it/s]


ROUGE RESULTS
rouge1: 0.3784
rouge2: 0.2030
rougeL: 0.3768
rougeLsum: 0.3765
